<a href="https://colab.research.google.com/github/malak-dotcom/life-expectancy-data-processing/blob/main/randomforestment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [ ]:
df=pd.read_csv('waze_dataset.csv')
df.head()

,ID,label,sessions,drives,total_sessions,n_days_after_onboarding,total_navigations_fav1,total_navigations_fav2,driven_km_drives,duration_minutes_drives,activity_days,driving_days,device
0,0,retained,283,226,296.748273,2276,208,0,2628.845068,1985.775061,28,19,Android
1,1,retained,133,107,326.896596,1225,19,64,13715.920550,3160.472914,13,11,iPhone
2,2,retained,114,95,135.522926,2651,0,0,3059.148818,1610.735904,14,8,Android
3,3,retained,49,40,67.589221,15,322,7,913.591123,587.196542,7,3,iPhone
4,4,retained,84,68,168.247020,1562,166,5,3950.202008,1219.555924,27,18,Android


In [ ]:
df.shape

(14999, 13)

In [ ]:
df.isnull().sum()


,0
ID,0
label,700
sessions,0
drives,0
total_sessions,0
n_days_after_onboarding,0
total_navigations_fav1,0
total_navigations_fav2,0
driven_km_drives,0
duration_minutes_drives,0


In [ ]:
#nulls
df['label'].fillna(df['label'].mode()[0],inplace=True)
df.drop_duplicates(inplace=True)


In [ ]:
#label encoder
from sklearn.preprocessing import LabelEncoder
cat=['label','device']
encoder=LabelEncoder()
for i in cat:
    df[i]=encoder.fit_transform(df[i])

In [ ]:
#standerization
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
df[df.columns]=scaler.fit_transform(df[df.columns])

In [ ]:
#impalance
df['label'].value_counts(normalize=True)*100


,proportion
label,
0.451090,83.092206
-2.216852,16.907794


In [ ]:
#data split
x=df.drop('label',axis=1)
y=df['label']
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [ ]:
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
unique_scaled_labels = y_train.unique()
sorted_scaled_labels = sorted(unique_scaled_labels)
y_train_discrete = pd.Categorical(y_train, categories=sorted_scaled_labels).codes
x_train_smote,y_train_smote=smote.fit_resample(x_train,y_train_discrete)

In [ ]:
#randomforestment
rf=RandomForestClassifier()
rf.fit(x_train_smote,y_train_smote)
y_pred=rf.predict(x_test)
unique_scaled_labels_test = y_test.unique()
sorted_scaled_labels_test = sorted(unique_scaled_labels_test)
y_test_discrete = pd.Categorical(y_test, categories=sorted_scaled_labels_test).codes

print(accuracy_score(y_test_discrete,y_pred))

0.7723333333333333


In [ ]:
#adaboost
from sklearn.ensemble import AdaBoostClassifier
ada_model = AdaBoostClassifier(
    n_estimators=100,
    learning_rate=1.0,
    random_state=42
)

unique_scaled_labels_train = y_train.unique()
sorted_scaled_labels_train = sorted(unique_scaled_labels_train)
y_train_discrete_ada = pd.Categorical(y_train, categories=sorted_scaled_labels_train).codes

unique_scaled_labels_test = y_test.unique()
sorted_scaled_labels_test = sorted(unique_scaled_labels_test)
y_test_discrete_ada = pd.Categorical(y_test, categories=sorted_scaled_labels_test).codes

# Train
ada_model.fit(x_train, y_train_discrete_ada)

# Prediction
y_pred = ada_model.predict(x_test)

# Evaluation
print("Accuracy:", accuracy_score(y_test_discrete_ada, y_pred))
print("Precision:", precision_score(y_test_discrete_ada, y_pred))
print("Recall:", recall_score(y_test_discrete_ada, y_pred))
print("F1 Score:", f1_score(y_test_discrete_ada, y_pred))



Accuracy: 0.8343333333333334
Precision: 0.844222833562586
Recall: 0.9823929571828731
F1 Score: 0.9080821157758461


In [ ]:
# Gradient Boosting
from sklearn.ensemble import GradientBoostingClassifier

gb_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)

# Convert labels to discrete integers
unique_scaled_labels_train = y_train.unique()
sorted_scaled_labels_train = sorted(unique_scaled_labels_train)
y_train_discrete_gb = pd.Categorical(
    y_train,
    categories=sorted_scaled_labels_train
).codes

unique_scaled_labels_test = y_test.unique()
sorted_scaled_labels_test = sorted(unique_scaled_labels_test)
y_test_discrete_gb = pd.Categorical(
    y_test,
    categories=sorted_scaled_labels_test
).codes

# Train
gb_model.fit(x_train, y_train_discrete_gb)

# Prediction
y_pred = gb_model.predict(x_test)

# Evaluation
print("Accuracy:", accuracy_score(y_test_discrete_gb, y_pred))
print("Precision:", precision_score(y_test_discrete_gb, y_pred))
print("Recall:", recall_score(y_test_discrete_gb, y_pred))
print("F1 Score:", f1_score(y_test_discrete_gb, y_pred))

Accuracy: 0.8343333333333334
Precision: 0.8423392612859097
Recall: 0.985594237695078
F1 Score: 0.908353309976028


In [ ]:
# XGBoost
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)

# Convert labels to discrete integers
unique_scaled_labels_train = y_train.unique()
sorted_scaled_labels_train = sorted(unique_scaled_labels_train)
y_train_discrete_xgb = pd.Categorical(
    y_train,
    categories=sorted_scaled_labels_train
).codes

unique_scaled_labels_test = y_test.unique()
sorted_scaled_labels_test = sorted(unique_scaled_labels_test)
y_test_discrete_xgb = pd.Categorical(
    y_test,
    categories=sorted_scaled_labels_test
).codes

# Train
xgb_model.fit(x_train, y_train_discrete_xgb)

# Prediction
y_pred = xgb_model.predict(x_test)

# Evaluation
print("Accuracy:", accuracy_score(y_test_discrete_xgb, y_pred))
print("Precision:", precision_score(y_test_discrete_xgb, y_pred))
print("Recall:", recall_score(y_test_discrete_xgb, y_pred))
print("F1 Score:", f1_score(y_test_discrete_xgb, y_pred))

Accuracy: 0.8336666666666667
Precision: 0.8455425017277125
Recall: 0.9791916766706683
F1 Score: 0.907472649731133
